In [1]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import tensorflow as tf
from tensorflow.keras.regularizers import L2
from tensorflow.keras.models import Sequential
from tensorflow.keras.preprocessing.text import Tokenizer
from tensorflow.keras.callbacks import EarlyStopping,ReduceLROnPlateau
from tensorflow.keras.utils import pad_sequences
from tensorflow.keras.layers import Dense, LSTM, Dropout,Embedding,SimpleRNN,GRU,SpatialDropout1D,Bidirectional
from tensorflow.keras import regularizers

In [ ]:
df=pd.read_csv(r'dataset\Combined Data.csv')

In [3]:
df.head()

,Unnamed: 0,statement,status
0,0,oh my gosh,Anxiety
1,1,"trouble sleeping, confused mind, restless hear...",Anxiety
2,2,"All wrong, back off dear, forward doubt. Stay ...",Anxiety
3,3,I've shifted my focus to something else but I'...,Anxiety
4,4,"I'm restless and restless, it's been a month n...",Anxiety


In [4]:
df=df.drop('Unnamed: 0',axis=1)

In [5]:
df.head()

,statement,status
0,oh my gosh,Anxiety
1,"trouble sleeping, confused mind, restless hear...",Anxiety
2,"All wrong, back off dear, forward doubt. Stay ...",Anxiety
3,I've shifted my focus to something else but I'...,Anxiety
4,"I'm restless and restless, it's been a month n...",Anxiety


In [6]:
df['status'].value_counts()

status
Normal                  16351
Depression              15404
Suicidal                10653
Anxiety                  3888
Bipolar                  2877
Stress                   2669
Personality disorder     1201
Name: count, dtype: int64

In [7]:
df.isnull().sum()

statement    362
status         0
dtype: int64

In [8]:
df_new=df.dropna()

In [9]:
df_new.isnull().sum()

statement    0
status       0
dtype: int64

In [10]:
X=df_new.drop(['status'],axis=1)

In [11]:
Y=df_new['status']

In [12]:
X_new=X.to_numpy().tolist()

In [13]:
X_new

[['oh my gosh'],
 ['trouble sleeping, confused mind, restless heart. All out of tune'],
 ['All wrong, back off dear, forward doubt. Stay in a restless and restless place'],
 ["I've shifted my focus to something else but I'm still worried"],
 ["I'm restless and restless, it's been a month now, boy. What do you mean?"],
 ['every break, you must be nervous, like something is wrong, but what the heck'],
 ['I feel scared, anxious, what can I do? And may my family or us be protected :)'],
 ["Have you ever felt nervous but didn't know why?"],
 ["I haven't slept well for 2 days, it's like I'm restless. why huh :([]."],
 ["I'm really worried, I want to cry."],
 ["always restless every night, even though I don't know why, what's wrong. strange."],
 ["I'm confused, I'm not feeling good lately. Every time I want to sleep, I always feel restless"],
 ['sometimes what is needed when there is a problem is to laugh until you forget that there is a problem, when you remember it, you feel restless like t

In [14]:
X_texts_new = np.array(X_new).ravel().tolist()

In [15]:
X_texts_new

['oh my gosh',
 'trouble sleeping, confused mind, restless heart. All out of tune',
 'All wrong, back off dear, forward doubt. Stay in a restless and restless place',
 "I've shifted my focus to something else but I'm still worried",
 "I'm restless and restless, it's been a month now, boy. What do you mean?",
 'every break, you must be nervous, like something is wrong, but what the heck',
 'I feel scared, anxious, what can I do? And may my family or us be protected :)',
 "Have you ever felt nervous but didn't know why?",
 "I haven't slept well for 2 days, it's like I'm restless. why huh :([].",
 "I'm really worried, I want to cry.",
 "always restless every night, even though I don't know why, what's wrong. strange.",
 "I'm confused, I'm not feeling good lately. Every time I want to sleep, I always feel restless",
 'sometimes what is needed when there is a problem is to laugh until you forget that there is a problem, when you remember it, you feel restless like that well, it turns out th

In [16]:
type(Y)

pandas.core.series.Series

In [17]:
from sklearn.preprocessing import OneHotEncoder

In [18]:
Y_reshaped=Y.to_numpy().reshape(-1,1)
encoder=OneHotEncoder(sparse_output=False)
Y_encoded=encoder.fit_transform(Y_reshaped)

In [19]:
(Y_encoded)

array([[1., 0., 0., ..., 0., 0., 0.],
       [1., 0., 0., ..., 0., 0., 0.],
       [1., 0., 0., ..., 0., 0., 0.],
       ...,
       [1., 0., 0., ..., 0., 0., 0.],
       [1., 0., 0., ..., 0., 0., 0.],
       [1., 0., 0., ..., 0., 0., 0.]], shape=(52681, 7))

In [20]:
tokenizer=Tokenizer(num_words=20000,oov_token="<OOV>")

In [21]:
from sklearn.model_selection import train_test_split
X_train,X_test,Y_train,Y_test=train_test_split(X_texts_new,Y_encoded,test_size=0.2,random_state=39)

In [22]:
tokenizer.fit_on_texts(X_train)

In [47]:
my_dict=tokenizer.word_index
next((k for k, v in my_dict.items() if v == 55000), None)


"i'llbe"

In [24]:
Y_test.shape,Y_train.shape,

((10537, 7), (42144, 7))

In [25]:
train_sequenced_text=tokenizer.texts_to_sequences(X_train)
test_sequenced_text=tokenizer.texts_to_sequences(X_test)

In [26]:
sequenced_text=tokenizer.texts_to_sequences(X_texts_new)

In [27]:
len(train_sequenced_text)

42144

In [28]:
sequenced_text

[[601, 6, 5076],
 [958, 658, 1005, 262, 1248, 296, 32, 43, 9, 8845],
 [32, 278, 92, 141, 2306, 578, 1152, 352, 16, 7, 1248, 4, 1248, 285],
 [208, 7321, 6, 691, 3, 101, 177, 17, 77, 95, 421],
 [77, 1248, 4, 1248, 179, 40, 7, 288, 55, 1358, 37, 18, 28, 385],
 [116, 430, 28, 761, 26, 829, 24, 101, 12, 278, 17, 37, 5, 3534],
 [2, 29, 249, 363, 37, 47, 2, 18, 4, 374, 6, 130, 30, 321, 26, 5938],
 [13, 28, 151, 186, 829, 17, 447, 36, 96],
 [2, 911, 1378, 176, 19, 222, 191, 179, 24, 77, 1248, 96, 3394],
 [77, 59, 421, 2, 31, 3, 384],
 [105, 1248, 116, 246, 50, 239, 2, 167, 36, 96, 1211, 278, 1337],
 [77, 1005, 77, 10, 109, 106, 501, 116, 56, 2, 31, 3, 204, 2, 105, 29, 1248],
 [237,
  37,
  12,
  458,
  45,
  68,
  12,
  7,
  356,
  12,
  3,
  1115,
  254,
  28,
  781,
  14,
  68,
  12,
  7,
  356,
  45,
  28,
  362,
  8,
  28,
  29,
  1248,
  24,
  14,
  176,
  8,
  1395,
  43,
  14,
  2,
  95,
  13,
  7,
  279,
  642,
  2061,
  2061,
  2061],
 [38, 21, 488, 12, 28],
 [237,
  179,
  127,
  238

In [29]:
sent_max_len=max(len(sent) for sent in sequenced_text)

In [30]:
sent_max_len

6300

In [29]:
train_padded_text=pad_sequences(train_sequenced_text,maxlen=100)
test_padded_text=pad_sequences(test_sequenced_text,maxlen=100)

In [30]:
print(train_padded_text)

[[   0    0    0 ...    4  279   56]
 [   0    0    0 ...    6 1503 1025]
 [   0    0    0 ...    8   76   14]
 ...
 [   0    0    0 ... 2637   23    1]
 [  69   27    3 ...   46   48   92]
 [   2   13   21 ...   15   95  162]]


In [31]:
dim=100
voc_size=len(tokenizer.word_index)+1
voc_size

55689

In [34]:
model=Sequential([
        Embedding(20000,dim),
        SpatialDropout1D(0.5),                   # better than Dropout for embeddings
        Bidirectional(GRU(32, recurrent_dropout=0.3,kernel_regularizer=regularizers.l2(1e-5))),   # single LSTM layer, not stacked
        Dropout(0.4),
        Dense(32, activation='relu', kernel_regularizer=regularizers.l2(1e-4)),
        Dropout(0.3),
        Dense(7, activation='softmax')
    ])

In [35]:
model.compile(optimizer='adam',loss='categorical_crossentropy',metrics=['accuracy'])

In [36]:
model.summary()

Model: "sequential"

┏━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━┓
┃ Layer (type)                    ┃ Output Shape           ┃       Param # ┃
┡━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━┩
│ embedding (Embedding)           │ ?                      │   0 (unbuilt) │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ spatial_dropout1d               │ ?                      │             0 │
│ (SpatialDropout1D)              │                        │               │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ bidirectional (Bidirectional)   │ ?                      │   0 (unbuilt) │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dropout (Dropout)               │ ?                      │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dense (Dense)                   │ ?                      │   0 (unbuilt) │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dropout_1 (Dropout)             │ ?                      │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dense_1 (Dense)                 │ ?                      │   0 (unbuilt) │
└─────────────────────────────────┴────────────────────────┴───────────────┘

 Total params: 0 (0.00 B)

 Trainable params: 0 (0.00 B)

 Non-trainable params: 0 (0.00 B)

In [37]:
stopping_callback=EarlyStopping(monitor='val_loss',patience=4  ,restore_best_weights=True)
reduce_lr = ReduceLROnPlateau(monitor='val_loss', factor=0.5, patience=2, min_lr=1e-6)

In [ ]:
history=model.fit(train_padded_text,Y_train, validation_data=(test_padded_text,Y_test),epochs=50,batch_size=32,callbacks=[stopping_callback,reduce_lr],shuffle=True)

In [ ]:
y_pred=model.predict(test_padded_text)

330/330 ━━━━━━━━━━━━━━━━━━━━ 4s 12ms/step


In [ ]:
y_pred.shape

(10537, 7)

In [ ]:
sample_text=["""We're officially making r/gamesonreddit open for direct posting! We want to trial providing a space that allows developers to share their creations, updates to their games, or key events happening. 

We’ll evaluate how successful it is at this over the next several weeks. Feel free to share feedback and ideas in the comments as well. 

Introducing new rules: 

To ensure a topical and balanced environment, we're implementing a few new rules related to staying on topic and overpromotion.

Stay on topic: This community is focused on discussion around games built on Reddit’s Developer Platform. Please keep posts and comments relevant! 

No overpromotion: Please refrain from excessive overpromotion or spamming your game. We want genuine engagement and discussion, not just repeated self-promotion. While we encourage developers to share their work, please be mindful of this rule and focus on quality contributions and meaningful interactions.."""]
seq = tokenizer.texts_to_sequences(sample_text)

# Pad it to same length as training
padded = pad_sequences(seq, maxlen=100)

In [ ]:
seq

[[1954,
  2262,
  287,
  1155,
  1,
  538,
  19,
  3031,
  985,
  80,
  31,
  3,
  3944,
  4802,
  7,
  928,
  14,
  4709,
  1,
  3,
  591,
  194,
  11356,
  6783,
  3,
  194,
  822,
  30,
  2283,
  1629,
  660,
  6671,
  6716,
  52,
  1215,
  8,
  12,
  34,
  21,
  115,
  5,
  302,
  786,
  300,
  29,
  555,
  3,
  591,
  2659,
  4,
  1338,
  16,
  5,
  1535,
  54,
  176,
  9164,
  220,
  2155,
  3,
  4120,
  7,
  1,
  4,
  5623,
  1557,
  1954,
  1,
  7,
  165,
  220,
  2155,
  900,
  3,
  1093,
  27,
  1739,
  4,
  1,
  352,
  27,
  1739,
  21,
  1104,
  12,
  2259,
  27,
  3096,
  174,
  822,
  2112,
  27,
  1,
  7763,
  4289,
  258,
  171,
  1399,
  4,
  1535,
  3854,
  49,
  1,
  258,
  13017,
  62,
  3240,
  1,
  30,
  14495,
  127,
  805,
  80,
  31,
  1982,
  6870,
  4,
  3096,
  10,
  20,
  4749,
  244,
  4415,
  198,
  80,
  3178,
  1,
  3,
  591,
  194,
  93,
  258,
  26,
  7498,
  9,
  21,
  3008,
  4,
  691,
  27,
  2368,
  14363,
  4,
  2225,
  2763]]

In [ ]:
print(padded)

[[  555     3   591  2659     4  1338    16     5  1535    54   176  9164
    220  2155     3  4120     7     1     4  5623  1557  1954     1     7
    165   220  2155   900     3  1093    27  1739     4     1   352    27
   1739    21  1104    12  2259    27  3096   174   822  2112    27     1
   7763  4289   258   171  1399     4  1535  3854    49     1   258 13017
     62  3240     1    30 14495   127   805    80    31  1982  6870     4
   3096    10    20  4749   244  4415   198    80  3178     1     3   591
    194    93   258    26  7498     9    21  3008     4   691    27  2368
  14363     4  2225  2763]]


In [ ]:
# Model prediction
pred = model.predict(padded)

pred_class = np.argmax(pred, axis=1)[0]
print("Predicted class index:", pred_class)


1/1 ━━━━━━━━━━━━━━━━━━━━ 0s 38ms/step
Predicted class index: 5


In [ ]:
pred_label = encoder.inverse_transform(pred)
print("Predicted label:", pred_label[0][0])

Predicted label: Stress
